## SCOS Migration Output
**Source File**: `00_orchestrator.ipynb`  
**Migrated on**: 2026-04-28  
**Conversion Tool**: Snowpark Connect (SCOS)

### Changes Applied
- SparkSession creation replaced with `snowpark_connect.init_spark_session()`
- PySpark imports annotated with SCOS compatibility comments
- Incompatible patterns flagged with `# SCOS-EWI:` markers

### Known Limitations
- Review `# SCOS-EWI:` markers for manual conversion required


# 00 — Tasty Bytes Consulting — Pipeline Orchestrator

**Tasty Bytes Consulting** is a fictional wealth management firm managing ~$2.5B
across institutional clients, family offices, and high-net-worth individuals.

This notebook orchestrates the full five-stage PySpark demo pipeline in order:

| Stage | Notebook | Topic |
|-------|----------|-------|
| 1 | `01_data_ingestion_schema.ipynb` | Schema enforcement, type casting, null handling |
| 2 | `02_scd_patterns.ipynb` | Slowly Changing Dimensions on `dim_clients` |
| 3 | `03_aggregations_pivot_windows.ipynb` | Aggregations, pivot tables, window functions |
| 4 | `04_functional_pipeline.ipynb` | Functional composition pipeline on transactions |
| 5 | `05_date_utilities_sql_interop.ipynb` | Date utilities and Spark SQL interoperability |

All stages share a common data module (`demo_data.py`) that defines the
Tasty Bytes Consulting reference data: clients, assets, portfolios,
transactions, prices, and FX rates.

In [ ]:
import os
import sys
import subprocess
import time
from pathlib import Path
import snowpark_connect

# Ensure demo_data.py is importable from this directory
BASE_DIR = Path(".").resolve()
sys.path.insert(0, str(BASE_DIR))

# Ordered list of pipeline stage notebooks
PIPELINE_NOTEBOOKS = [
    BASE_DIR / "01_data_ingestion_schema.ipynb",
    BASE_DIR / "02_scd_patterns.ipynb",
    BASE_DIR / "03_aggregations_pivot_windows.ipynb",
    BASE_DIR / "04_functional_pipeline.ipynb",
    BASE_DIR / "05_date_utilities_sql_interop.ipynb",
]

print(f"Pipeline base directory : {BASE_DIR}")
print(f"Notebooks to execute    : {len(PIPELINE_NOTEBOOKS)}")

## 1. Pre-flight checks

Verify that all pipeline notebooks and the shared data module exist before
attempting execution.

In [ ]:
all_ok = True

# Check shared data module
demo_data_path = BASE_DIR / "demo_data.py"
if demo_data_path.exists():
    print(f"  [OK] demo_data.py")
else:
    print(f"  [MISSING] demo_data.py  <-- required by all stages")
    all_ok = False

# Check each notebook
for nb in PIPELINE_NOTEBOOKS:
    if nb.exists():
        size_kb = round(nb.stat().st_size / 1024, 1)
        print(f"  [OK] {nb.name}  ({size_kb} KB)")
    else:
        print(f"  [MISSING] {nb.name}")
        all_ok = False

if all_ok:
    print("\nAll files present. Ready to execute.")
else:
    print("\nOne or more files are missing. Resolve before running.")

## 2. Execution engine

`run_notebook()` executes a single notebook via `jupyter nbconvert --execute`
and writes output back into the same file.  Each notebook manages its own
`SparkSession` lifecycle (create at start, stop at end) so stages do not
share session state — keeping them independently runnable and debuggable.

If `jupyter` is not available on `PATH`, a fallback message is printed
with the equivalent shell command.

In [ ]:
def run_notebook(notebook_path: Path, timeout_seconds: int = 300) -> dict:
    """
    Execute *notebook_path* using jupyter nbconvert.

    Returns a result dict with keys:
        name        : str    notebook filename
        success     : bool
        elapsed_s   : float  wall-clock seconds
        returncode  : int
        stderr_tail : str    last 400 chars of stderr on failure
    """
    t0 = time.time()
    cmd = [
        sys.executable, "-m", "jupyter", "nbconvert",
        "--to", "notebook",
        "--execute",
        "--inplace",
        f"--ExecutePreprocessor.timeout={timeout_seconds}",
        str(notebook_path),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = round(time.time() - t0, 1)
    return {
        "name":        notebook_path.name,
        "success":     proc.returncode == 0,
        "elapsed_s":   elapsed,
        "returncode":  proc.returncode,
        "stderr_tail": proc.stderr[-400:] if proc.returncode != 0 else "",
    }


def print_result(r: dict) -> None:
    """Pretty-print a run_notebook result."""
    status = "OK     " if r["success"] else "FAILED "
    print(f"  [{status}] {r['name']}  ({r['elapsed_s']}s)")
    if not r["success"] and r["stderr_tail"]:
        print(f"           stderr: ...{r['stderr_tail']}")


print("Execution engine ready.")

## 3. Run the pipeline

Execute all five stages sequentially.  The overall pipeline typically
takes 2-4 minutes on a laptop with `local[*]` Spark.

> **Tip**: to run a single stage independently, open its notebook and
> run all cells — each notebook is fully self-contained.

In [ ]:
pipeline_start = time.time()
results = []

print("=" * 60)
print("  Tasty Bytes Consulting — PySpark Pipeline")
print("=" * 60)

for notebook in PIPELINE_NOTEBOOKS:
    stage_num = notebook.stem.split("_")[0]  # e.g. "01"
    print(f"\nStage {stage_num} — executing {notebook.name} ...")
    result = run_notebook(notebook)
    results.append(result)
    print_result(result)
    if not result["success"]:
        print(f"  Pipeline halted at stage {stage_num}. Fix the error above and re-run.")
        break

total_elapsed = round(time.time() - pipeline_start, 1)
passed = sum(1 for r in results if r["success"])

print("\n" + "=" * 60)
print(f"  {passed}/{len(PIPELINE_NOTEBOOKS)} stages completed  ({total_elapsed}s total)")
print("=" * 60)

## 4. Execution summary table

Summarise timing and status for every stage that was attempted.

In [ ]:
#EWI: [FIXED] SPRKCNTPY1000 => PySpark import detected — verify compatibility with Snowpark Connect
#EWI: [FIXED] SPRKCNTPY1000 => create_spark_session() replaced with snowpark_connect.init_spark_session()
# SCOS-EWI: [SPRKCNTPY1000] PySpark import — verify Snowpark Connect compatibility
from demo_data import COMPANY_NAME  # create_spark_session removed — replaced by snowpark_connect.init_spark_session()
# SCOS-EWI: [SPRKCNTPY1000] PySpark import — verify Snowpark Connect compatibility
from pyspark.sql import Row

# SCOS-EWI: [SPRKCNTPY1001] Session replaced with Snowpark Connect
spark = snowpark_connect.init_spark_session()
#EWI: [FIXED] SPRKCNTPY4000 => setLogLevel uses SparkContext which is not available in Snowpark Connect
# SCOS-EWI: [SPRKCNTPY4000] setLogLevel removed — SparkContext not available in SCOS
print(f"Pipeline: {COMPANY_NAME}\n")

summary_rows = [
    Row(
        stage=str(i + 1),
        notebook=r["name"],
        status="OK" if r["success"] else "FAILED",
        elapsed_s=r["elapsed_s"],
    )
    for i, r in enumerate(results)
]

summary_df = spark.createDataFrame(summary_rows)
print("=== Pipeline execution summary ===")
summary_df.show(truncate=False)

## 5. Tasty Bytes Consulting — data asset overview

Load all shared reference datasets and display key statistics to confirm
the data foundation is intact across all pipeline stages.

In [ ]:
from demo_data import load_all
# SCOS-EWI: [SPRKCNTPY1000] PySpark import — verify Snowpark Connect compatibility
from pyspark.sql import functions as F

data = load_all(spark)

print("=== Data asset inventory ===")
for name, df in data.items():
    print(f"  {name:15s}: {df.count():>4} rows   cols: {df.columns}")

print("\n=== Client AUM by type ===")
(
    data["clients"]
    .groupBy("client_type")
    .agg(
        F.count("*").alias("clients"),
        F.round(F.sum("aum_usd") / 1e6, 1).alias("total_aum_m_usd"),
    )
    .orderBy("client_type")
).show()

print("=== Portfolio count and NAV by strategy ===")
(
    data["portfolios"]
    .groupBy("strategy")
    .agg(
        F.count("*").alias("portfolios"),
        F.round(F.sum("nav_usd") / 1e6, 1).alias("total_nav_m_usd"),
    )
    .orderBy("strategy")
).show()

In [ ]:
print("=== Transaction volume by type ===")
(
    data["transactions"]
    .groupBy("txn_type")
    .agg(
        F.count("*").alias("trade_count"),
        F.round(
            F.sum(F.col("quantity") * F.col("price") * F.col("fx_rate_to_usd")),
            2,
        ).alias("total_notional_usd"),
    )
    .orderBy("txn_type")
).show()

print("=== Asset universe by class ===")
(
    data["assets"]
    .groupBy("asset_class")
    .agg(
        F.count("*").alias("instruments"),
        F.collect_set("currency").alias("currencies"),
    )
    .orderBy("asset_class")
).show(truncate=False)

print("=== FX rates as of last available date ===")
# SCOS-EWI: [SPRKCNTPY1000] PySpark import — verify Snowpark Connect compatibility
from pyspark.sql.window import Window

latest_fx_window = Window.partitionBy("base_ccy").orderBy(F.desc("rate_date"))
(
    data["fx_rates"]
    .withColumn("rn", F.row_number().over(latest_fx_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .orderBy("base_ccy")
).show()

## 6. Integrated cross-stage query

Demonstrate that data from multiple stages can be combined in a single
query: join transactions → assets → portfolios to produce a full
trade-level enriched view — the kind of output that downstream
risk or reporting systems would consume.

In [ ]:
txn  = data["transactions"]
ast  = data["assets"]
prt  = data["portfolios"]
cli  = data["clients"]

enriched_trades = (
    txn
    .join(ast.select("asset_id", "ticker", "asset_class", "sector"), on="asset_id", how="left")
    .join(prt.select("portfolio_id", "portfolio_name", "strategy", "client_id"), on="portfolio_id", how="left")
    .join(cli.select("client_id", "client_name", "relationship_manager"), on="client_id", how="left")
    .withColumn("notional_usd", F.round(F.col("quantity") * F.col("price") * F.col("fx_rate_to_usd"), 2))
    .withColumn("txn_date", F.to_date(F.col("txn_date"), "yyyy-MM-dd"))
)

enriched_trades.createOrReplaceTempView("vw_enriched_trades")
# SCOS: createOrReplaceTempView supported in Snowpark Connect

print("=== Sample enriched trade ledger ===")
enriched_trades.select(
    "txn_id", "txn_date", "txn_type", "ticker",
    "asset_class", "portfolio_name", "strategy",
    "client_name", "notional_usd",
).orderBy("txn_date").show(10, truncate=False)

In [ ]:
# Relationship manager book of business — total notional traded YTD
rm_summary = spark.sql("""
    SELECT
        relationship_manager,
        strategy,
        COUNT(DISTINCT portfolio_id)          AS portfolios,
        COUNT(*)                              AS trades,
        ROUND(SUM(notional_usd) / 1e6, 2)    AS notional_m_usd
    FROM vw_enriched_trades
    WHERE txn_type IN ('buy', 'sell')
    GROUP BY relationship_manager, strategy
    ORDER BY relationship_manager, notional_m_usd DESC
""")

print("=== RM book of business by strategy (buy/sell notional) ===")
rm_summary.show(truncate=False)

# Monthly trade activity across all portfolios
monthly_activity = spark.sql("""
    SELECT
        DATE_FORMAT(txn_date, 'yyyy-MM')          AS year_month,
        COUNT(*)                                   AS trades,
        COUNT(DISTINCT portfolio_id)               AS active_portfolios,
        ROUND(SUM(CASE WHEN txn_type='buy'  THEN notional_usd ELSE 0 END) / 1e6, 2) AS buy_notional_m,
        ROUND(SUM(CASE WHEN txn_type='sell' THEN notional_usd ELSE 0 END) / 1e6, 2) AS sell_notional_m
    FROM vw_enriched_trades
    GROUP BY DATE_FORMAT(txn_date, 'yyyy-MM')
    ORDER BY year_month
""")

print("=== Monthly trading activity ===")
monthly_activity.show(truncate=False)

## 7. Pipeline complete

All five stages have been executed and the integrated cross-stage view
confirms data consistency across the Tasty Bytes Consulting dataset.

**What each stage demonstrated:**

1. **Data Ingestion** — StructType schemas, type casting, null handling,
   reject-row separation, per-column null profiling
2. **SCD Patterns** — SHA-256 hash change detection, Type I / Type II
   update logic, audit columns on dimension and fact tables
3. **Aggregations & Windows** — `groupBy` / `pivot`, `rank` / `dense_rank`,
   running totals, `lag` / `lead`, `rollup`
4. **Functional Pipeline** — `transform()` chaining, parameterised closures,
   config-driven `when` chains, join enrichment, inline validation
5. **Date Utilities & SQL** — date parsing, time-period classification,
   Python UDFs, `createOrReplaceTempView`, mixed SQL ↔ DataFrame workflows

All notebooks and helper modules (`helper_methods.py`, `data_transformations.py`,
`dataframe_validators.py`) are self-contained and can be run independently
or as part of this orchestrated pipeline.

In [ ]:
#EWI: [FIXED] SPRKCNTPY1000 => spark.stop() removed — session lifecycle managed by Snowpark Connect
# spark.stop()  # SCOS: session lifecycle managed by Snowpark Connect
print("Orchestrator complete.")